# TP02 — OOP
## Exercise 1: Build the `CSVReader` Class

### Objective
This exercise practices the basic concepts of Object-Oriented Programming (OOP):

- **Class** A class is a template for creating objects.
- **Constructor (`__init__`)** A constructor is the special method that runs automatically when an object is created.
- **Methods** A method is a function that belongs to a class.
- **Encapsulation** Encapsulation means keeping related data and functions together inside one class.

The `CSVReader` class will keep all CSV-reading operations inside one reusable class.

## 1 Import pandas

We use **pandas** because `pandas.read_csv()` can load a CSV file into a DataFrame.

In [55]:
import pandas as pd

## 2 — Create the `CSVReader` Class

The class contains:

- `__init__(file_path)` → stores the CSV file path. A constructor is a special method that runs automatically when you create a new object.
- `read()` → reads the CSV file using pandas.
- `preview(n)` → prints the first `n` rows.

`self.data` stores the loaded DataFrame so that other methods can reuse it.

In [56]:
class CSVReader:
    """A simple reusable class for reading and previewing CSV files."""

    def __init__(self, file_path):
        """Store the path of the CSV file."""
        self.file_path = file_path
        self.data = None

    def read(self):
        """Read the CSV file and store it in self.data."""
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        """Print the first n rows of the loaded CSV file."""
        if self.data is None:
            self.read()

        print(self.data.head(n))

## 3 — Create an Object

The CSV file is in the same folder as this notebook, so we pass its filename to `CSVReader`.

In [57]:
reader = CSVReader("sample_data.csv")

## 4 — Read the CSV File

Calling `read()` loads the CSV file and returns a pandas DataFrame.

In [58]:
df = reader.read()
df

,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


## 5 — Preview the First 5 Rows

The `preview()` method displays the top rows of the dataset.  
By default, `n=5`.

In [59]:
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## 6 — Preview a Different Number of Rows

Because `preview(n)` accepts an argument, we can choose how many rows to display.

In [60]:
reader.preview(3)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN


## Result

The `CSVReader` class successfully:

1. Stores the CSV file path.
2. Reads the file with pandas.
3. Stores the DataFrame inside the object.
4. Previews any requested number of rows.

This completes **Exercise 1**.

## Exercise 2: Apply Pattern for Missing Values Handling 

### Objective

- an abstract class **MissingValueStrategy**
- three subclasses:
  - **DropMissing**
  - **FillMean**
  - **FillMode**
- a **DataCleaner** class that uses the selected strategy

The goal is to practice Strategy Pattern, inheritance, and polymorphism.

### Import Library 

`from abc import ABC, abstractmethod` is used to create an abstract class.

In [61]:
from abc import ABC, abstractmethod

### Create the abstract strategy

In [62]:
class MissingValueStrategy(ABC):

    @abstractmethod
    def handle(self, df):
        pass

- `MissingValueStrategy` a general strategy for handling missing values.
- `@abstractmethod` Every child class must create its own handle() method.

### Create DropMissing

In [63]:
class DropMissing(MissingValueStrategy):

    def handle(self, df):
        return df.dropna()

What does `dropna()` do?
It removes rows containing missing values.

### Create `FillMean`

In [64]:
class FillMean(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        numeric_columns = df.select_dtypes(
            include="number"
        ).columns

        for column in numeric_columns:
            df[column] = df[column].fillna(
                df[column].mean()
            )

        return df

`df = df.copy`This makes a copy.

Why?

Because we do not want to modify the original DataFrame directly.

`numeric_columns = df.select_dtypes(
    include="number"
).columns`
 This finds columns containing numbers.

### Create `FillMode`

In [65]:
class FillMode(MissingValueStrategy):

    def handle(self, df):
        df = df.copy()

        for column in df.columns:

            if df[column].isnull().any():

                mode = df[column].mode()

                if not mode.empty:
                    df[column] = df[column].fillna(
                        mode[0]
                    )

        return df

This checks every column wheather its column has maissing value or not, if it has fill it by `Mode`, but if it does not has missing value skip.


### Create `DataCleaner`

In [66]:
class DataCleaner:

    def __init__(self, strategy):
        self.strategy = strategy

    def set_strategy(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

### Testing

In [67]:
# Drop Missing
cleaner = DataCleaner(DropMissing())

result = cleaner.clean(df)

print("\n--- Drop Missing ---")
print(result)


# Fill Mean
cleaner.set_strategy(FillMean())

result = cleaner.clean(df)

print("\n--- Fill Mean ---")
print(result)


# Fill Mode
cleaner.set_strategy(FillMode())

result = cleaner.clean(df)

print("\n--- Fill Mode ---")
print(result)


--- Drop Missing ---
   id   name   age  height_cm  weight_kg      city  score
0   1  Alice  29.0      165.0       68.0  New York   85.0
4   5    Eva  27.0      160.0       55.0  New York   88.0

--- Fill Mean ---
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0

--- Fill Mode ---
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3 

# Exercise 3: Add Decorators for Logging and Timing

**Objectives**
- Learn the **Decorator Pattern** to add reusable behaviour (logging, timing).
- Understand how decorators support the **Open/Closed Principle**: classes stay *closed for modification* but *open for extension*.

**Concept:** a decorator is a function that takes a function, wraps it with extra behaviour, and returns the wrapper. The original method body is never edited.

In [68]:
import sys
import time
import logging
import functools
import pandas as pd

# Configure logging once so the decorators print to the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,   # same stream as print() so output stays in order
    force=True,
)
logger = logging.getLogger("pipeline")

## Step 1 — Create the two decorators

- `@log_action` → logs when a method **starts** and **finishes** (and logs errors if it fails).
- `@log_time` → measures and prints the **execution time**.

`functools.wraps` keeps the original function's name and docstring, so `func.__name__` still says `read_csv` instead of `wrapper`.

In [69]:
def log_action(func):
    """Log when a function/method starts and finishes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"▶ START  {func.__qualname__}")
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"✖ FAILED {func.__qualname__}: {e}")
            raise
        logger.info(f"✔ FINISH {func.__qualname__}")
        return result
    return wrapper


def log_time(func):
    """Measure and print how long a function/method takes."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"⏱ {func.__qualname__} took {elapsed * 1000:.3f} ms")
        return result
    return wrapper

### Quick test on a plain function

In [70]:
@log_action
@log_time
def slow_add(a, b):
    time.sleep(0.2)
    return a + b

print("Result:", slow_add(2, 3))
print("Name preserved by functools.wraps:", slow_add.__name__)

19:05:57 | INFO | ▶ START  slow_add
⏱ slow_add took 200.440 ms
19:05:57 | INFO | ✔ FINISH slow_add
Result: 5
Name preserved by functools.wraps: slow_add


## Step 2 — Apply the decorators to `CSVReader` and `DataCleaner`

The class logic is exactly the same as before — the only change is the decorator lines above each method. That is the Open/Closed Principle in action.

In [71]:
class CSVReader:
    """Responsible only for loading CSV data."""

    def __init__(self, file_path):
        self.file_path = file_path

    @log_action
    @log_time
    def read_csv(self):
        df = pd.read_csv(self.file_path)
        print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
        return df


class DataFrameCleaner:
    """Responsible only for cleaning data."""

    def __init__(self, df):
        self.df = df.copy()

    @log_action
    @log_time
    def show_missing(self):
        missing = self.df.isnull().sum()
        print("Missing values per column:")
        print(missing[missing > 0])
        return missing

    @log_action
    @log_time
    def fill_missing_with_mean(self):
        num_cols = self.df.select_dtypes(include="number").columns
        self.df[num_cols] = self.df[num_cols].fillna(self.df[num_cols].mean().round(1))
        return self.df

    @log_action
    @log_time
    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        print(f"Removed {before - len(self.df)} duplicate rows")
        return self.df

## Step 3 — Run the pipeline and watch the logs

In [72]:
reader = CSVReader("sample_data.csv")
raw_df = reader.read_csv()
raw_df

19:05:57 | INFO | ▶ START  CSVReader.read_csv
Loaded 7 rows × 7 columns
⏱ CSVReader.read_csv took 2.337 ms
19:05:57 | INFO | ✔ FINISH CSVReader.read_csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [73]:
cleaner = DataFrameCleaner(raw_df)
cleaner.show_missing()
cleaner.fill_missing_with_mean()
clean_df = cleaner.remove_duplicates()
clean_df

19:05:57 | INFO | ▶ START  DataFrameCleaner.show_missing
Missing values per column:
age          3
height_cm    1
weight_kg    1
score        2
dtype: int64
⏱ DataFrameCleaner.show_missing took 1.985 ms
19:05:57 | INFO | ✔ FINISH DataFrameCleaner.show_missing
19:05:57 | INFO | ▶ START  DataFrameCleaner.fill_missing_with_mean
⏱ DataFrameCleaner.fill_missing_with_mean took 4.225 ms
19:05:57 | INFO | ✔ FINISH DataFrameCleaner.fill_missing_with_mean
19:05:57 | INFO | ▶ START  DataFrameCleaner.remove_duplicates
Removed 0 duplicate rows
⏱ DataFrameCleaner.remove_duplicates took 1.721 ms
19:05:57 | INFO | ✔ FINISH DataFrameCleaner.remove_duplicates


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


In [74]:
print("Missing values after cleaning:", clean_df.isnull().sum().sum())

Missing values after cleaning: 0


## Step 4 — Open/Closed Principle without touching the class

We can even add timing/logging to a class **we are not allowed to edit** by wrapping its method from the outside:

In [75]:
class Reporter:
    def summary(self, df):
        return df.describe().round(2)

# Extend behaviour from the outside — Reporter's source is unchanged
Reporter.summary = log_action(log_time(Reporter.summary))

Reporter().summary(clean_df)

19:05:57 | INFO | ▶ START  Reporter.summary
⏱ Reporter.summary took 12.006 ms
19:05:57 | INFO | ✔ FINISH Reporter.summary


,id,age,height_cm,weight_kg,score
count,7.00,7.00,7.00,7.00,7.00
mean,4.00,30.23,167.00,70.00,86.00
std,2.16,2.41,5.29,10.47,5.45
min,1.00,27.00,160.00,55.00,75.00
25%,2.50,29.50,163.50,64.00,85.50
50%,4.00,30.20,167.00,70.00,86.00
75%,5.50,30.20,170.00,76.00,89.00
max,7.00,35.00,175.00,85.00,92.00


## Summary

| Decorator | What it adds | Applied to |
|---|---|---|
| `@log_action` | Start / finish (and error) log messages | `read_csv`, `show_missing`, `fill_missing_with_mean`, `remove_duplicates` |
| `@log_time` | Execution time in ms | same methods |

**Why this follows Open/Closed:** logging and timing were added without modifying the bodies of `CSVReader` or `DataCleaner`. The decorators are reusable on any function, and removing them is as easy as deleting one line.

**Stacking order:** with `@log_action` on top of `@log_time`, `log_action` is the outer wrapper — its START prints first and FINISH prints last, and the timer measures only the method itself.

# Exercise 4: Factory Pattern for Data Transformations

The factory creates transformation objects from names, while every transformation follows the same `DataTransform` interface.

In [76]:
class DataTransform(ABC):
    @abstractmethod
    # Purpose: Forces subclasses to implement a method. Prevents instantiation of the base class.
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a transformed copy of the input DataFrame."""
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Normalize numeric columns to the range [0, 1]."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include='number').columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if pd.notna(minimum) and pd.notna(maximum) and maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            elif pd.notna(minimum):
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    # text column: applies strip().lower() to string values
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include=['object', 'string']).columns:
            result[column] = result[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return result


class TransformFactory:
    #One registry maps string names to classes, letting you create transformations dynamically without hardcoding imports or if/else chains.
    _TRANSFORMS = {
        'normalize': NormalizeColumns,
        'remove_duplicates': RemoveDuplicates,
        'duplicates': RemoveDuplicates,
        'standardize_text': StandardizeText,
        'text': StandardizeText,
    }

    @classmethod
    # Purpose: Factory method to create instances of DataTransform subclasses based on a string name.
    def create(cls, transform_name: str) -> DataTransform:
        key = transform_name.strip().lower()
        try:
            return cls._TRANSFORMS[key]()
        # Handle the case where the provided transform name is not in the registry.
        except KeyError as error:
            supported = ', '.join(sorted(cls._TRANSFORMS))
            raise ValueError(
            # Provide a helpful error message with the list of supported transformation names.
                f"Unknown transformation '{transform_name}'. Supported names: {supported}"
            ) from error

In [77]:
# Use the fully cleaned DataFrame produced by Exercise 2 (FillMean strategy).
# Use the fully cleaned DataFrame produced by Exercise 3.
cleaned_input = clean_df.copy()

assert cleaned_input.isna().sum().sum() == 0

normalized_df = TransformFactory.create('normalize').apply(cleaned_input)
deduplicated_df = TransformFactory.create('remove_duplicates').apply(cleaned_input)
standardized_df = TransformFactory.create('text').apply(cleaned_input)

print('Normalized numeric columns:')
display(normalized_df)

print('Duplicate rows removed:')
display(deduplicated_df)

print('Text columns standardized:')
display(standardized_df)

pipeline = ['text', 'normalize', 'remove_duplicates']
transformed_df = cleaned_input.copy()

for transform_name in pipeline:
    transformed_df = TransformFactory.create(transform_name).apply(transformed_df)

print('Composed pipeline result:')
display(transformed_df)



Normalized numeric columns:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,Alice,0.250,0.333333,0.433333,New York,0.588235
1,0.166667,Bob,0.400,0.800000,0.500000,Los Angeles,0.882353
2,0.333333,Charlie,1.000,0.533333,0.566667,Chicago,0.647059
3,0.500000,David,0.400,0.466667,0.833333,Houston,0.000000
4,0.666667,Eva,0.000,0.000000,0.000000,New York,0.764706
5,0.833333,Frank,0.375,1.000000,1.000000,Los Angeles,0.647059
6,1.000000,Grace,0.400,0.133333,0.166667,Chicago,1.000000


Duplicate rows removed:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,30.2,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,86.0
3,4,David,30.2,167.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.2,162.0,60.0,Chicago,92.0


Text columns standardized:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.0,165.0,68.0,new york,85.0
1,2,bob,30.2,172.0,70.0,los angeles,90.0
2,3,charlie,35.0,168.0,72.0,chicago,86.0
3,4,david,30.2,167.0,80.0,houston,75.0
4,5,eva,27.0,160.0,55.0,new york,88.0
5,6,frank,30.0,175.0,85.0,los angeles,86.0
6,7,grace,30.2,162.0,60.0,chicago,92.0


Composed pipeline result:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.250,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.400,0.800000,0.500000,los angeles,0.882353
2,0.333333,charlie,1.000,0.533333,0.566667,chicago,0.647059
3,0.500000,david,0.400,0.466667,0.833333,houston,0.000000
4,0.666667,eva,0.000,0.000000,0.000000,new york,0.764706
5,0.833333,frank,0.375,1.000000,1.000000,los angeles,0.647059
6,1.000000,grace,0.400,0.133333,0.166667,chicago,1.000000


## Exercise 5 · A complete CSV cleaning pipeline

Build a reusable workflow with four design patterns:

| Pattern | Responsibility |
| :--- | :--- |
| **Template Method** | `DataPipeline.run()` fixes the order: load → clean → transform → save. |
| **Strategy** | `DataCleaner` delegates missing-value handling to an interchangeable strategy. |
| **Factory** | `TransformFactory` creates transformations from configuration names. |
| **Decorator** | `@log_action` and `@log_time` provide optional execution diagnostics. |

Run all cells in order. The example writes `dirty_sample.csv` and `cleaned_sample.csv`
in the notebook's working directory. Verbose diagnostics are disabled by default.

### Pipeline template · Template Method Pattern

The base class defines the workflow once. Concrete subclasses implement the four
steps. `run()` returns the pipeline so callers can inspect its result and summary.

In [78]:
class DataPipeline(ABC):
    @log_action
    @log_time
    def run(self):
        """Execute the fixed load → clean → transform → save sequence."""
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self

    @abstractmethod
    def load(self) -> None:
        ...

    @abstractmethod
    def clean(self) -> None:
        ...

    @abstractmethod
    def transform(self) -> None:
        ...

    @abstractmethod
    def save(self) -> None:
        ...

### Concrete CSV pipeline

Implement the four template steps using the reader, cleaner, and transformation factory.

In [79]:
class CSVDataPipeline(DataPipeline):
    def __init__(self, input_path, output_path, missing_strategy=None, transforms=None):
        self.input_path = input_path
        self.output_path = output_path
        self.cleaner = DataCleaner(missing_strategy)
        self.transforms = tuple(transforms) if transforms is not None else ()
        self.df = None

    def load(self):
            self.df = CSVReader(self.input_path).read_csv()

    def clean(self):
        self.df = self.cleaner.clean(self.df)

    def transform(self):
        for name in self.transforms:
            transformer = TransformFactory.create(name)
            self.df = transformer.apply(self.df)

    def save(self):
        self.df.to_csv(self.output_path, index=False)

### Create the sample dataset

The five input rows contain inconsistent headers and text casing, one duplicate
Alice record, and three missing cells: age, salary, and department.

In [80]:
from pathlib import Path
from IPython.display import display

In [81]:
INPUT_PATH = Path("dirty_sample.csv")
OUTPUT_PATH = Path("cleaned_sample.csv")

raw_df = pd.DataFrame({
    " Full Name ": [" Alice ", "BOB", "Charlie ", " Alice ", "David"],
    " Age ": [25.0, None, 30.0, 25.0, 40.0],
    " Salary ": [50000.0, 60000.0, None, 50000.0, 75000.0],
    " Department ": [" HR ", "engineering", "FINANCE", " HR ", None],
})
raw_df.to_csv(INPUT_PATH, index=False)

### Run the pipeline

Run the four steps and compare the original and cleaned data using standard pandas tables.

`FillMean` fills age with **30** and salary with **58,750** before deduplication.
David's department remains missing because it is not numeric.

In [82]:
pipeline = CSVDataPipeline(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    missing_strategy=FillMean(),
    transforms=["normalize", "standardize_text", "remove_duplicates"],
).run()

cleaned_df = pipeline.df
print("Before cleaning:")
display(raw_df)
print("After cleaning:")
display(cleaned_df)


19:05:58 | INFO | ▶ START  DataPipeline.run
19:05:58 | INFO | ▶ START  CSVReader.read_csv
Loaded 5 rows × 4 columns
⏱ CSVReader.read_csv took 2.284 ms
19:05:58 | INFO | ✔ FINISH CSVReader.read_csv
⏱ DataPipeline.run took 11.059 ms
19:05:58 | INFO | ✔ FINISH DataPipeline.run
Before cleaning:


,Full Name,Age,Salary,Department
0,Alice,25.0,50000.0,HR
1,BOB,NaN,60000.0,engineering
2,Charlie,30.0,NaN,FINANCE
3,Alice,25.0,50000.0,HR
4,David,40.0,75000.0,NaN


After cleaning:


,Full Name,Age,Salary,Department
0,alice,0.000000,0.00,hr
1,bob,0.333333,0.40,engineering
2,charlie,0.333333,0.35,finance
3,david,1.000000,1.00,NaN
